# 01. 복잡한 이미지 프롬프트의 구조화

목표: 정보 밀도가 높은 이미지 요청을 목표, 캔버스, 패널, 고정 문자열, 스타일, 금지 조건으로 나눕니다. API나 외부 패키지는 사용하지 않습니다.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Panel:
    name: str
    region: str
    content: str
    exact_text: list[str] = field(default_factory=list)

@dataclass
class ImageBrief:
    goal: str
    audience: str
    canvas: str
    panels: list[Panel]
    style: list[str]
    forbidden: list[str]

## 예제 brief

고정 문자열은 설명과 분리합니다. 모델이 내용을 임의로 바꾸는 위험을 줄이고, 생성 후 OCR 정답표로도 재사용하기 위해서입니다.

In [ ]:
brief = ImageBrief(
    goal="확산 모델의 복원 과정을 설명하는 한국어 교육 인포그래픽",
    audience="생성형 AI를 처음 배우는 대학생",
    canvas="A4 세로, 상단 20% 제목, 중앙 60% 흐름도, 하단 20% 주의사항",
    panels=[
        Panel("1단계", "중앙 왼쪽", "원본 이미지에 노이즈 추가", ["노이즈 추가"]),
        Panel("2단계", "중앙 가운데", "신경망이 노이즈를 예측", ["노이즈 예측"]),
        Panel("3단계", "중앙 오른쪽", "여러 번 반복하여 이미지 복원", ["반복 복원"]),
    ],
    style=["흰 배경", "남색 제목", "높은 명암 대비", "최소한의 장식"],
    forbidden=["가짜 로고", "읽을 수 없는 장식 문자", "출처 없는 통계"],
)

In [ ]:
def render_prompt(brief: ImageBrief) -> str:
    # 각 섹션에 책임을 하나씩 부여하면 긴 프롬프트의 충돌을 찾기 쉽습니다.
    lines = [
        f"[목표]\n{brief.goal}",
        f"[독자]\n{brief.audience}",
        f"[캔버스와 구획]\n{brief.canvas}",
    ]
    panel_lines = []
    for panel in brief.panels:
        labels = ", ".join(f'\"{text}\"' for text in panel.exact_text)
        panel_lines.append(f"- {panel.name} / {panel.region}: {panel.content}; 고정 문자열: {labels}")
    lines.append("[패널]\n" + "\n".join(panel_lines))
    lines.append("[스타일]\n" + ", ".join(brief.style))
    lines.append("[금지]\n" + ", ".join(brief.forbidden))
    return "\n\n".join(lines)

prompt = render_prompt(brief)
print(prompt)

## 길이와 충돌 검사

아래 token 수는 정확한 Qwen tokenizer 결과가 아니라 공백과 문자 수를 이용한 거친 추정치입니다. 실제 한도 검사는 서비스 tokenizer 또는 API 응답을 사용해야 합니다.

In [ ]:
def rough_token_estimate(text: str) -> int:
    # 한글은 공백 단위와 실제 token 수가 다르므로 안전 여유를 둔 추정만 제공합니다.
    non_space_chars = sum(not ch.isspace() for ch in text)
    whitespace_units = len(text.split())
    return max(whitespace_units, round(non_space_chars / 2))

def lint_brief(brief: ImageBrief) -> list[str]:
    problems = []
    if not brief.panels:
        problems.append("패널이 없습니다.")
    all_labels = [text for panel in brief.panels for text in panel.exact_text]
    if len(all_labels) != len(set(all_labels)):
        problems.append("중복 고정 문자열이 있습니다.")
    if any(not panel.region for panel in brief.panels):
        problems.append("위치가 지정되지 않은 패널이 있습니다.")
    if rough_token_estimate(render_prompt(brief)) > 4_000:
        problems.append("4.5K 한도에 가까울 수 있으므로 실제 tokenizer로 확인하세요.")
    return problems

print("rough tokens:", rough_token_estimate(prompt))
print("lint:", lint_brief(brief) or "문제 없음")

다음 실습에서는 이 구조화된 prompt를 공식 API 메시지 형태로 감싸되 실제 요청은 보내지 않습니다.